# Load ProductSummary into Cosmos DB (reverse ETL)

Part 3 of the Cosmos DB + AI layer demo. Workspace `cosmos db`, default lakehouse `retail_ai_lh`.

1. Aggregate the Cosmos mirror (`SampleData` shortcut) into the gold Delta table `gold_product_review_summary`.
2. Upsert each row into the Cosmos container `ProductSummary` (`/productId` partition key, `id = productId`), so the app reads it with a ~1 RU point read.
3. Verify with a point read.

Auth uses the notebook identity (`notebookutils.credentials.getToken`); no keys. Run after the gold refresh, then run `operational-api/jobs/precompute-insights.js`.


In [ ]:
# RETIRED. Gold is now built only from the KPI registry by sync_gold_to_cosmos
# (operational-api/registry/metrics.json -> generated SQL). This notebook used to overwrite gold with
# its own hand-written definitions, so it now stops here instead of silently replacing registry metrics.
raise RuntimeError("Retired: run sync_gold_to_cosmos instead. Metric definitions live in the KPI registry.")

In [ ]:
%pip install "azure-cosmos>=4.14.0" -q

In [ ]:
import base64, json, time, datetime, decimal
from azure.core.credentials import AccessToken, TokenCredential
from azure.cosmos import CosmosClient, PartitionKey, ThroughputProperties

ENDPOINT = "https://ae268e38-3ca9-4db0-815c-2c9c808d346b.zae.sql.cosmos.fabric.microsoft.com"

class FabricTokenCredential(TokenCredential):
    def get_token(self, *scopes, **kwargs):
        tok = notebookutils.credentials.getToken("https://cosmos.azure.com/")
        payload = tok.split(".")[1]; payload += "=" * (-len(payload) % 4)
        return AccessToken(tok, json.loads(base64.urlsafe_b64decode(payload))["exp"])

db = CosmosClient(ENDPOINT, credential=FabricTokenCredential()).get_database_client("retail_cosmos")
print("Connected:", [c["id"] for c in db.list_containers()])


In [ ]:
container = db.create_container_if_not_exists(
    id="ProductSummary",
    partition_key=PartitionKey(path="/productId"),
    offer_throughput=ThroughputProperties(auto_scale_max_throughput=1000),  # Fabric requires autoscale
)

def to_json_safe(v):
    if isinstance(v, (datetime.date, datetime.datetime)): return v.isoformat()
    if isinstance(v, decimal.Decimal): return float(v)
    return v

rows = spark.table("gold_product_review_summary").collect()
t0 = time.time(); ru = 0.0
for r in rows:
    doc = {k: to_json_safe(v) for k, v in r.asDict().items()}
    doc["id"] = doc["productId"]          # id = partition key, which enables cheap point reads
    doc["docType"] = "productSummary"
    container.upsert_item(doc)
    ru += float(container.client_connection.last_response_headers["x-ms-request-charge"])
print(f"Loaded {len(rows)} docs in {time.time()-t0:.1f}s, {ru:.0f} RU")


In [ ]:
pid = rows[0]["productId"]
t0 = time.perf_counter()
item = container.read_item(item=pid, partition_key=pid)
ms = (time.perf_counter() - t0) * 1000
print(f"{item['productName']}: {item['reviewCount']} reviews, avg {item['avgStars']}  "
      f"→ {ms:.0f} ms, {container.client_connection.last_response_headers['x-ms-request-charge']} RU")
